In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
import sys
import os

# Agregar la carpeta src al path para usar nuestro pipeline
sys.path.append(os.path.abspath(os.path.join('..')))
from src.preprocessing.pipeline import PipelinePLN

print(" Librerías importadas correctamente")
print(f"GPU disponible: {torch.cuda.is_available()}")

 Librerías importadas correctamente
GPU disponible: False


In [2]:
# 1. Se carga el dataset de 100 registros
ruta_dataset = '../data/labeled/dataset_ejemplo.csv'
df = pd.read_csv(ruta_dataset)
print(f" Dataset cargado: {len(df)} registros")

# 2. Inicializar nuestro pipeline del Sprint 1
pipeline = PipelinePLN(idioma='es')

# 3. Aplicar preprocesamiento a todos los textos
print(" Preprocesando textos (esto puede tomar unos segundos)...")
df['texto_procesado'] = df['texto'].apply(lambda x: pipeline.procesar_texto_completo(x)['texto_procesado'])

# Mostrar distribución y primeros resultados
print("\n Distribución de clases:")
print(df['etiqueta'].value_counts().sort_index())
print("\n Ejemplo de preprocesamiento:")
display(df[['texto', 'texto_procesado', 'etiqueta']].head(3))

 Dataset cargado: 100 registros


INFO: Pipeline inicializado exitosamente para idioma: es


 Preprocesando textos (esto puede tomar unos segundos)...

 Distribución de clases:
etiqueta
0    34
1    33
2    33
Name: count, dtype: int64

 Ejemplo de preprocesamiento:


,texto,texto_procesado,etiqueta
0,Me siento muy bien con mi progreso en el semes...,sentir bien progreso semestre clase interesante,0
1,"Hoy entendí por fin los árboles binarios, me s...",hoy entender fin árbol binario sentir satisfec...,0
2,"La clase de programación estuvo excelente, log...",programación excelente logré resolver ejercicio,0


In [3]:
# Se divide en entrenamiento (80%) y prueba (20%)
# Con 100 registros, tendremos 80 para entrenar y 20 para evaluar
train_df, eval_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['etiqueta']  # Mantiene la proporción de clases
)

print(f" Dataset dividido:")
print(f"   - Entrenamiento: {len(train_df)} registros")
print(f"   - Evaluación: {len(eval_df)} registros")

# Carga tokenizador de BETO (la primera vez lo descarga de internet, aproximadamnte400MB)
modelo_nombre = "dccuchile/bert-base-spanish-wwm-uncased"
print(f"\n Descargando tokenizador de {modelo_nombre}...")
tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)

# Función para tokenizar los datos
def tokenize_function(examples):
    return tokenizer(examples["texto_procesado"], truncation=True, max_length=128)

# Convierte a formato Dataset de HuggingFace
train_dataset = Dataset.from_pandas(train_df[['texto_procesado', 'etiqueta']])
eval_dataset = Dataset.from_pandas(eval_df[['texto_procesado', 'etiqueta']])

# Renombra columna 'etiqueta' a 'labels' (requerido por HuggingFace)
train_dataset = train_dataset.rename_column("etiqueta", "labels")
eval_dataset = eval_dataset.rename_column("etiqueta", "labels")

# Tokeniza en lote
print(" Tokenizando datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Data collator para agrupar los batches dinámicamente
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(" Datasets preparados y tokenizados")

 Dataset dividido:
   - Entrenamiento: 80 registros
   - Evaluación: 20 registros

 Descargando tokenizador de dccuchile/bert-base-spanish-wwm-uncased...


INFO: HTTP Request: HEAD https://huggingface.co/dccuchile/bert-base-spanish-wwm-uncased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dccuchile/bert-base-spanish-wwm-uncased/d1c9c4565c9d6731e57ed7f027b802697bad861e/config.json?%2Fdccuchile%2Fbert-base-spanish-wwm-uncased%2Fresolve%2Fmain%2Fconfig.json=&etag=%22e82679f58a4a5f8fb5e31ebf5de8417e6e2ffb8d%22 "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/dccuchile/bert-base-spanish-wwm-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dccuchile/bert-base-spanish-wwm-uncased/d1c9c4565c9d6731e57ed7f027b802697bad861e/tokenizer_config.json?%2Fdccuchile%2Fbert-base-spanish-wwm-uncased%2Fresolve%2Fmain%2Ftokenizer_config.json=&etag=%22823d1878940e4bd96d9f786d438d6c95ec74c670%22 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api

 Tokenizando datasets...


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

 Datasets preparados y tokenizados


In [4]:
# Carga modelo base con 3 clases
print("Cargando modelo BETO para fine-tuning...")
model = AutoModelForSequenceClassification.from_pretrained(
    modelo_nombre, 
    num_labels=3,
    id2label={0: "Positivo/Tranquilo", 1: "Neutro", 2: "Negativo/Estresado"},
    label2id={"Positivo/Tranquilo": 0, "Neutro": 1, "Negativo/Estresado": 2}
)

# Configuración de entrenamiento optimizada para 100 registros
training_args = TrainingArguments(
    output_dir="./resultados_beto",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",  
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
)

# Inicializa Trainer con el nombre de argumento actualizado
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer, 
    data_collator=data_collator,
)

print("Iniciando entrenamiento...")
trainer.train()

INFO: HTTP Request: HEAD https://huggingface.co/dccuchile/bert-base-spanish-wwm-uncased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dccuchile/bert-base-spanish-wwm-uncased/d1c9c4565c9d6731e57ed7f027b802697bad861e/config.json?%2Fdccuchile%2Fbert-base-spanish-wwm-uncased%2Fresolve%2Fmain%2Fconfig.json=&etag=%22e82679f58a4a5f8fb5e31ebf5de8417e6e2ffb8d%22 "HTTP/1.1 200 OK"


Cargando modelo BETO para fine-tuning...


INFO: HTTP Request: HEAD https://huggingface.co/dccuchile/bert-base-spanish-wwm-uncased/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/dccuchile/bert-base-spanish-wwm-uncased "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/dccuchile/bert-base-spanish-wwm-uncased/commits/main "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/dccuchile/bert-base-spanish-wwm-uncased/discussions?p=0 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

INFO: HTTP Request: GET https://huggingface.co/api/models/dccuchile/bert-base-spanish-wwm-uncased/commits/refs%2Fpr%2F3 "HTTP/1.1 200 OK"
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSIN

...

D:\WORKGRADE\prototipo-estres-academico-pln\venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

D:\WORKGRADE\prototipo-estres-academico-pln\venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=50, training_loss=0.7232271575927735, metrics={'train_runtime': 63.9935, 'train_samples_per_second': 6.251, 'train_steps_per_second': 0.781, 'total_flos': 1952794897200.0, 'train_loss': 0.7232271575927735, 'epoch': 5.0})

In [5]:
# Evalua el modelo con el conjunto de prueba
print(" Evaluando modelo con el conjunto de prueba...")
results = trainer.evaluate()

print("\n Métricas de evaluación finales:")
for key, value in results.items():
    if isinstance(value, float):
        print(f"   - {key}: {value:.4f}")
    else:
        print(f"   - {key}: {value}")

# Guarda el modelo entrenado y el tokenizador
modelo_guardado = "../src/models/beto_finetuned"
trainer.save_model(modelo_guardado)
tokenizer.save_pretrained(modelo_guardado)
print(f"\n Modelo guardado exitosamente en: {modelo_guardado}")

 Evaluando modelo con el conjunto de prueba...


D:\WORKGRADE\prototipo-estres-academico-pln\venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
0.432049,0.538958,5



 Métricas de evaluación finales:
   - eval_loss: 0.5390


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 Modelo guardado exitosamente en: ../src/models/beto_finetuned
